# TSFresh-Features aus rohem und DyCVDA-gefiltertem TEP-Signal

Drittes Schwester-Notebook zu `TSFresh_PCA_DyCA.ipynb` und
`TSFresh_DPCA_CVA_ICA.ipynb`: identische dreiphasige Pipeline (Extraktion +
Selektion auf Train, billige Test-Extraktion, LazyClassifier), aber mit
**DyCVDA** als Projektion (*Dynamical and Canonical Variate Dissimilarity
Analysis*, Wu et al., IEEE Trans. Autom. Sci. Eng. 23, 2026, S. 9560–9570).
`raw` läuft als gemeinsamer Anker mit — dadurch sind die Ergebnisse aller drei
Notebooks direkt aneinander ausrichtbar.

## Was DyCVDA ist

DyCVDA verkettet zwei Stufen, die es in der Notebook-Familie beide schon
einzeln gibt:

1. **DyCA** (identisch zur `dyca_*`-Projektion des ersten
   Schwester-Notebooks): zerlegt die 52 Prozessvariablen in *n*
   deterministische Amplituden y(t), deren zeitliche Entwicklung ein
   gekoppeltes lineares ODE-System optimal erfüllt (Paper Gl. 5–11).
2. **CVA-Dissimilarität** (verwandt mit der `cva_*`-Projektion des zweiten
   Schwester-Notebooks, aber auf den DyCA-Amplituden statt auf den Rohdaten,
   mit Horizont *s* statt p = f = 1 und mit Zukunftsterm): aus
   Vergangenheits- und Zukunftsstapeln y_p(t), y_f(t) wird per kanonischer
   Korrelationsanalyse die Projektion (J, L, Λ) bestimmt und daraus die
   **Dissimilaritätskanäle** d(t) = J·y_p(t) − Λ_r·L·y_f(t)
   (Paper Gl. 12–15).

Im Paper ist d(t) der Rohstoff eines skalaren Monitoring-Index D(t)
(Mahalanobis-Distanz mit KDE-Kontrollgrenze, Gl. 16–19) zur
Fehler-**Detektion**. Hier interessiert Fehler-**Klassifikation**: statt des
skalaren Index geht die r-kanalige d(t)-Zeitreihe in TSFresh — genau wie bei
allen anderen Projektionen der Familie. Details und die bewusste Abweichung
vom Paper (Fit pro Run statt Offline-Training auf Normalbetrieb) stehen im
Projektions-Abschnitt weiter unten.

## Die 7 Konfigurationen

| Konfiguration | Kanäle | DyCA (m, n) | Horizont s | Bemerkung |
|---|---|---|---|---|
| `raw` | 52 | — | — | keine Projektion; bitidentisch zu den Schwester-Notebooks |
| `dycvda_m3n6_s2_r6` | 6 | (3, 6) | 2 | kleine DyCA-Stufe |
| `dycvda_m4n8_s2_r8` | 8 | (4, 8) | 2 | mittlere DyCA-Stufe |
| `dycvda_m6n12_s2_r12` | 12 | (6, 12) | 2 | DyCA-Stufe = `dyca_m6_n12` des Schwester-Notebooks |
| `dycvda_m6n12_s2_r15` | 15 | (6, 12) | 2 | Paper-Setting für TEP: 12 det. Komponenten, s = 2, r = 15 |
| `dycvda_m6n12_s4_r12` | 12 | (6, 12) | 4 | längerer Horizont |
| `dycvda_m6n12_s8_r12` | 12 | (6, 12) | 8 | noch längerer Horizont |

Zusammen 52 + 65 = **117 Kanäle pro Run**. Alle vier Parameter (m, n, s, r)
stecken im Konfigurationsnamen und damit im Cache-Präfix — ein
Parameterwechsel erzeugt neue Konfigurationen, statt stumm fremde Chunks zu
laden (anders als `dpca_lags`/`cva_past` im Schwester-Notebook; Ausnahmen
sind nur `cva_ridge_rel` und `fix_signs`, die beide nicht im Namen stecken —
siehe Konfigurationszelle).

**Hinweis zur Verschachtelung.** Die CVA-Stufe ist in r verschachtelt wie PCA
in n: `dycvda_m6n12_s2_r15` enthält die 12 Kanäle von `…_r12` als exakte
Teilmenge, die drei zusätzlichen Kanäle sind die schwächsten kanonischen
Korrelationen. Ob die Top-100-Auswahl jenseits von Kanal 12 überhaupt
zugreift, zeigt der Vergleich der beiden Auswahl-JSONs — dasselbe Phänomen,
das bei `pca_6`/`pca_8` zu byteidentischen Ergebnissen geführt hat. Die s-
und (m, n)-Varianten sind dagegen **echt** verschiedene Projektionen.

## Kanallängen — kleine, dokumentierte Abweichung

Die Stapelbildung verkürzt die d(t)-Reihe auf 480 − 2s + 1 Samples: **477**
(s = 2), **473** (s = 4), **465** (s = 8); `raw` behält **480**. Das ist
unkritisch, weil die Regel „keine Längenartefakte" nur *innerhalb* einer
Konfiguration greifen muss: dort sind alle Runs und beide Splits exakt gleich
lang. Zwischen Konfigurationen werden Features nie verglichen — jede bekommt
ihre eigene Selektion und ihren eigenen Klassifizierer. `raw` bleibt bewusst
bei 480, damit die raw-Ergebnisse mit den Schwester-Notebooks bitidentisch
sind und der Cache geteilt werden kann (siehe unten).

## Gemeinsamer Cache mit den Schwester-Notebooks

`cache_dir` ist absichtlich **derselbe** (`tsfresh_cache` bzw.
`tsfresh_cache_smoke`): die `raw`-Konfiguration ist in allen drei Notebooks
bitidentisch (gleiche Vorverarbeitung, gleiche Länge, gleiches Chunking) —
vorhandene raw-Chunks und die raw-Top-`top_k`-Auswahl werden dadurch
**wiederverwendet**, was bei bereits gelaufenen Schwester-Notebooks rund
5,5 h spart. Zwei Konsequenzen:

- **`chunk_runs` muss auf 250 bleiben** (Wert der Schwester-Notebooks) —
  sonst würden deren Cache-Dateien mit falscher Chunk-Aufteilung
  interpretiert und Runs stumm verloren gehen.
- Die Ergebnis-CSV heißt hier `tsfresh_summary_dycvda.csv`, damit die
  Summary-CSVs der Schwester-Notebooks nicht überschrieben werden.

Das Präfix `dycvda_*` kollidiert mit nichts Bestehendem.

## Vorgehen (dreistufig)

**Phase A — Extraktion + Selektion auf TRAIN.** Je Konfiguration werden alle
TSFresh-Features auf allen Trainingsruns berechnet (`EfficientFCParameters`,
~780 Features je Kanal), daraus per Relevanztabelle die besten `top_k`
ausgewählt. Danach wird die volle Matrix sofort verworfen.

**Phase B — Testset.** `from_columns()` übersetzt die ausgewählten
Featurenamen zurück in `kind_to_fc_parameters`; auf dem Testset werden nur
diese Features berechnet. Die Selektion sieht das Testset nie.

**Phase C — LazyClassifier** je Konfiguration, dann Vergleich über alle 7.

## Laufzeit und Speicher — bitte vor dem Start lesen

Maßstab aus dem vollständigen Lauf des ersten Schwester-Notebooks (8 Kerne,
`EfficientFCParameters`, alle 500 Runs je Fault): ~6,5 min TSFresh-Zeit pro
Kanal in Phase A. Hier: `raw` aus dem Cache ≈ 0 h (sonst ~5,5 h), die 6
DyCVDA-Konfigurationen ≈ 65 Kanäle → Phase A grob **7 h**, dazu die
DyCA+CVA-Fits (6 × 10 500 Runs) mit grob **1 h** in Phase A und nochmals
~1 h in Phase B. Phase B insgesamt ~2–3 h, Phase C mit 5-facher CV ~1 h.
Zusammen Größenordnung **10–13 h** bei vorhandenem raw-Cache.

Speicher ist unkritischer als im ersten Schwester-Notebook: die größte neue
Konfiguration hat 15 Kanäle (~11 700 Features/Run); nur `raw` (40 404
Features/Run) bleibt groß — und kommt in der Regel aus dem Cache. Die
Mechanik (float32-Matrizen, Konfigurationen strikt nacheinander,
Relevanztests in Spaltenblöcken, Chunk-Cache) ist identisch.

**Der Cache macht den Lauf unterbrechbar.** Jeder Chunk landet als Pickle in
`cache_dir`; ein Neustart überspringt alles bereits Berechnete.

> **`smoke_test` steht in der Konfigurationszelle.** Mit `True` läuft die
> komplette Pipeline in wenigen Minuten auf wenigen Runs durch. Für den
> echten Lauf auf `False` (am besten über Nacht).

## Code-Struktur

Der gesamte Maschinenraum liegt im Paket [`tep/`](tep) und wird von allen
TEP-Notebooks geteilt. Dieses Notebook enthält nur noch, was es von
seinen Geschwistern unterscheidet: die Konfiguration, die Liste der
Projektions-Specs und die Aufrufe.

| Modul | Inhalt |
|---|---|
| `tep/core.py` | Spaltennamen, Splits, Cutoffs, Vorverarbeitung, lineare Algebra — geteilt mit `tep.eigen` |
| `tep/plotting.py` | Confusion-Matrix-Darstellung, ebenfalls geteilt |
| `tep/tsfresh/config.py` | `PipelineConfig` — alle Stellschrauben |
| `tep/tsfresh/projections.py` | Registry der Verfahren: `raw`, `pca`, `dyca`, `dpca`, `cva`, `ica`, `dycvda` |
| `tep/tsfresh/features.py` | Chunk-Cache-Extraktion, Feature-Ranking |
| `tep/tsfresh/pipeline.py` | Phase A / B / C |
| `tep/tsfresh/reporting.py` | Vergleichstabellen und Balkenplot |
| `tep/tsfresh/confusion.py` | Confusion-Matrizen und ihre drei Plots |

Eine Projektion ist ein `Projector` mit drei Angaben: wie sie heisst (der
Name ist zugleich Cache-Praefix), wie ihre Kanaele heissen und wie sie
rechnet. Ein eigenes Verfahren kommt über `tep.tsfresh.register(...)`
dazu, ohne dass hier etwas angefasst werden muss.

> Die Rechnungen sind zeilengetreu aus der früheren Notebook-Fassung
> uebernommen — Projektionen und Cache-Praefixe wurden vor dem Umbau
> ueber alle Konfigurationen und beide Skalierungsmodi als bitidentisch
> nachgewiesen. Der vorhandene Chunk-Cache bleibt damit gueltig.


In [ ]:
# ============================================================
# Imports - der gemeinsame Unterbau steckt im Paket tep
# ============================================================
# Wird tep/**.py bearbeitet, muss der Kernel neu gestartet werden;
# alternativ die beiden autoreload-Zeilen aktivieren.
# %load_ext autoreload
# %autoreload 2

# numpy/pandas werden hier nicht gebraucht, stehen aber fuer eigene
# Auswertungen am Ende des Notebooks bereit.
import numpy as np
import pandas as pd

from tep.tsfresh import (Pipeline, PipelineConfig, plot_confusion_detail,
                         plot_recall, versions)

print(versions())

In [ ]:
# ============================================================
# Konfiguration - die EINZIGE Stelle, an der geschraubt wird
# ============================================================
# Alle nicht gesetzten Felder stehen auf den Defaults aus
# tep/tsfresh/config.py (top_k=100, fc_mode='efficient',
# chunk_runs=250, run_length=480, lc_cv_folds=5, ...).
# smoke_test=True gibt einen winzigen Probelauf in einem
# eigenen Cache-Ordner - ueberschreibt also nichts.

# Jede Konfiguration ist ein Tupel (m, n, s, r):
#   m, n : DyCA-Stufe wie im Schwester-Notebook TSFresh_PCA_DyCA
#          (m lineare ODE-Komponenten, n deterministische Amplituden;
#          n = 2m ist der Grenzfall der Paketbedingung m >= n - m)
#   s    : Vergangenheits-/Zukunftshorizont der CVA-Stufe (Paper: s = 2)
#   r    : behaltene Dissimilaritaetskanaele d_1..d_r (r <= n*s)
# Alle vier Parameter stecken im Konfigurationsnamen (dycvda_m6n12_s2_r12)
# und damit im Cache-Praefix - ein Parameterwechsel kann hier KEINE
# fremden Chunks stumm weiterverwenden.
DYCVDA_CONFIGS = [
    (3, 6, 2, 6),       # kleine DyCA-Stufe
    (4, 8, 2, 8),       # mittlere DyCA-Stufe
    (6, 12, 2, 12),     # DyCA-Stufe = dyca_m6_n12 des Schwester-Notebooks
    (6, 12, 2, 15),     # Paper-Setting fuer TEP: 12 det. Komp., s=2, r=15
    (6, 12, 4, 12),     # laengerer Horizont
    (6, 12, 8, 12),     # noch laengerer Horizont
]

CFG = PipelineConfig(
    configs=([("raw",)]
             + [("dycvda", m, n, s, r) for (m, n, s, r) in DYCVDA_CONFIGS]),
    label="DyCVDA",
    summary_csv="tsfresh_summary_dycvda.csv",
    cm_pred_csv="tsfresh_cm_predictions_dycvda.csv",
    scaling_mode="global_mean",

    # Relative Ridge-Regularisierung der CVA-Stufe. Steckt - wie
    # fix_signs - NICHT im Cache-Namen: bei Aenderung vorher die
    # dycvda_*-Chunks aus dem Cache-Ordner loeschen.
    cva_ridge_rel=1e-6,
)

## Rohdaten laden

Identisch zu den Schwester-Notebooks: die Runs werden **einmal** in ein Dictionary
`{(faultNumber, simulationRun): Array}` gelesen und danach für alle 7
Konfigurationen wiederverwendet — die TEP-CSVs (besonders
`TEP_Faulty_Testing.csv` mit 3,4 GB) sollen nur ein Mal durch den Parser.

Speicher: mit `float32` und nur den benötigten Spalten sind das je ~1,05 GB
für Train und Test (beide Splits werden auf `run_length` gekürzt). Train und
Test werden **nie gleichzeitig** gehalten — Phase A braucht nur Train,
Phase B nur Test.

Sortiert wird pro Run (nicht global), weil ein globales `sort_values` über
10 Mio. Zeilen eine komplette Kopie anlegen würde. Die DyCA-Stufe
(Ableitungsschätzung) und die CVA-Stufe (Vergangenheits-/Zukunftsstapel)
brauchen die zeitliche Ordnung zwingend.

### `uniform_length` und `run_length`

Beide Schalter übernehmen unverändert die Begründung aus
`TSFresh_PCA_DyCA.ipynb`: längenabhängige TSFresh-Features (`length`,
`abs_energy`, `sum_values`, `count_above_mean`, …) würden sonst Fault 0
artifiziell abtrennen (innerhalb eines Splits) bzw. Train- und Test-Features
systematisch verschieden skalieren (zwischen den Splits: 480 vs. 800
Post-Fault-Samples). `run_length = 480` kürzt deshalb **jeden** Run auf die
ersten 480 Post-Fault-Samples = 24 h nach Fehlereintritt — in beiden Splits
dasselbe physikalische Fenster.

Dass die Projektion daraus config-spezifisch 465–480 Samples macht, ist
davon unberührt — innerhalb jeder Konfiguration bleiben alle Runs und beide
Splits exakt gleich lang (siehe Kanallängen-Abschnitt oben).

## Projektionen: roh und DyCVDA

Beide Varianten bekommen **dieselbe Vorverarbeitung**, gesteuert über
`scaling_mode` — zwischen den Konfigurationen unterscheidet sich wirklich nur
die Projektion. DyCVDA wird — wie alle Projektionen dieser Notebook-Familie —
**pro Run gefittet** (Begründung unten).

### `scaling_mode` — die DC-Problematik wird in der CVA-Stufe entschärft

Die DC-Problematik des ersten Schwester-Notebooks gilt für die DyCA-Stufe
hier unverändert: mit `"global_mean"` bleibt in jeder Spalte ein großer
Gleichanteil stehen, die DyCA-Amplituden sind DC-dominiert
(std/|mean| ~ 1e-5…1e-3). Zwei Dinge entschärfen das für DyCVDA:

- Die **CVA-Stufe zentriert** ihre Vergangenheits-/Zukunftsstapel explizit —
  der Gleichanteil fällt vor der Korrelationsrechnung wieder heraus. In
  float64 kostet das keine relevante Genauigkeit (nach Abzug von 3–5
  Größenordnungen DC bleiben ~11 signifikante Stellen).
- Die Dissimilaritätskanäle haben per Konstruktion **Kovarianz I − Λ_r²**
  (auf den Fit-Daten), sind also von der Skala der Amplituden unabhängig.

Was bleibt, ist der Einfluss des Gleichanteils auf die **Unterraumwahl der
DyCA-Stufe selbst** — welcher Unterraum als „deterministisch" erkannt wird,
hängt an der unzentrierten Korrelationsmatrix. **Aktuell ist `"global_mean"`
eingestellt** — konsistent mit den Schwester-Notebooks; der Vergleich mit dem
Scaler-Lauf (`tsfresh_cache_scaler`) zeigt, wie groß dieser Effekt ist.

### Fit pro Run — bewusste Abweichung vom Paper

Im Paper wird DyCVDA **offline** auf Normalbetriebsdaten trainiert (Θ‡, J,
L, Λ fest) und online werden Abweichungen vom Normalbetrieb detektiert. Hier
wird stattdessen pro Run gefittet, aus denselben Gründen wie bei allen
anderen Projektionen der Familie: Methode und Fit-Modus nicht vermischen
(pca/dyca/dpca/cva/ica sind alle pro Run gefittet), und DyCA sucht ohnehin
den Unterraum *dieses* Laufs.

Die Bedeutung von d(t) verschiebt sich dadurch: statt „Abweichung vom
Normalbetrieb" misst d(t), wie schlecht sich die DyCA-Dynamik **dieses Runs**
durch ihre eigene Vergangenheits-/Zukunfts-Korrelationsstruktur erklären
lässt — die Kanäle sind die am stärksten korrelierten Richtungen, d(t) deren
Residuum. TSFresh charakterisiert anschließend die *Form* dieser Residuen
(Autokorrelation, Spektrum, Ausreißerstatistik, …), und die Klassifikation
prüft, ob diese Form faultspezifisch ist. Die papiertreue Offline-Variante
steht als Erweiterung in der letzten Zelle.

### Definition (Paper Gl. 12–15)

Je Run, auf den n DyCA-Amplituden y(t):

- Stapel: y_p(t) = [y(t−1); …; y(t−s)], y_f(t) = [y(t); …; y(t+s−1)],
  beide zentriert (das Paper normalisiert die Daten vorab).
- CVA: H = Σ_ff^(−1/2) Σ_fp Σ_pp^(−1/2) = U Λ Vᵀ, dann
  J = V_rᵀ Σ_pp^(−1/2), L = U_rᵀ Σ_ff^(−1/2) (Ridge wie im CVA-Notebook).
- Kanäle: d(t) = J·y_p(t) − Λ_r·L·y_f(t), geordnet nach kanonischer
  Korrelation absteigend (`cvd1` = stärkste) — deterministisch, keine
  Ordnungswillkür wie bei ICA.

**Selbsttest der Implementierung:** auf den Fit-Daten gilt exakt
cov(d) = I − Λ_r² (ohne Ridge auf Maschinengenauigkeit verifiziert, mit
Ridge bis ~1e-6). Serienlänge 480 − 2s + 1 (siehe Kanallängen-Abschnitt).

**Vorzeichenkonvention.** Die d-Kanäle erben die Vorzeichen-Willkür der SVD
(u_i, v_i) → (−u_i, −v_i) — TSFresh-Features wie `mean`, `skewness` oder die
Steigung von `linear_trend` kippen mit. `fix_signs=True` dreht deshalb jeden
Kanal so, dass sein betragsmäßig größter Wert positiv ist — dieselbe
Konvention wie in den Schwester-Notebooks.

**Skip-Verhalten.** Die DyCA-Stufe kann an einzelnen Runs numerisch scheitern
(„Negative eigenvalues", wie im Schwester-Notebook); solche Runs werden für
die betroffene Konfiguration übersprungen, `restrict_to_common_runs` gleicht
die Run-Mengen in Phase C wieder an.

Bleibt die Einschränkung, dass pro Run gefittete Achsen in verschiedenen Runs
verschiedene physikalische Richtungen sind. Formstatistiken (Autokorrelation,
Spektrum, Entropie) bleiben vergleichbar; lageabhängige Features sind mit
Vorsicht zu lesen. Das ist derselbe Preis wie in den Schwester-Notebooks.

In [ ]:
# ============================================================
# Pipeline anlegen
# ============================================================
# Prueft die Specs (Rangbedingungen der Verfahren, doppelte Namen) und
# fittet bei scaling_mode="scaler" den StandardScaler auf dem
# Normalbetrieb. Danach steht der Umfang des Laufs im Klartext da.
pipe = Pipeline(CFG)
pipe.describe()

## Phase A — Extraktion und Selektion auf dem Trainingssatz

Identisch zu den Schwester-Notebooks. Je Konfiguration:

1. **Extrahieren** in Chunks à `chunk_runs` Runs. Jeder Chunk wird als
   `float32`-Pickle gecacht — ein Abbruch kostet höchstens den angefangenen
   Chunk. Für `raw` liegen die Chunks (und die Top-`top_k`-Auswahl) bei
   bereits gelaufenen Schwester-Notebooks schon im geteilten Cache.
2. **Selektieren:** Relevanztabelle in Spaltenblöcken (`block_cols`), damit
   die Roh-Konfiguration mit 40 404 Features nicht den Speicher sprengt.
3. **Reduzieren** auf `top_k` und die volle Matrix sofort freigeben.

**Zur Rangfolge.** Bei 10 500 Trainingsruns unterlaufen die p-Werte der
Signifikanztests reihenweise auf exakt 0.0 — nach p-Wert allein wären
hunderte Features gleichauf. Deshalb: `n_significant` (Zahl der Klassen, die
ein Feature signifikant trennt) absteigend, bei Gleichstand der
**ANOVA-F-Wert** — eine stetige Effektstärke, die nicht unterläuft.

Die Blockweise verschiebt die Benjamini-Hochberg-Korrektur minimal (sie sieht
je Block nur dessen p-Werte). Für eine *Rangfolge* ist das ohne Belang.

In [ ]:
# ============================================================
# PHASE A: Train extrahieren -> selektieren -> auf top_k reduzieren
# ============================================================
# Laeuft aus dem Chunk-Cache weiter, wenn schon Chunks vorhanden sind.
# Ergebnis liegt danach in pipe.train_top und pipe.top_names.
pipe.run_phase_a()

## Phase B — dieselben Features auf dem Testset

`from_columns()` übersetzt die ausgewählten Featurenamen zurück in ein
`kind_to_fc_parameters`-Dictionary. `extract_features` berechnet damit **nur**
diese Features — und auch nur auf den Kanälen, die in der Auswahl überhaupt
vorkommen. Deshalb ist Phase B um Größenordnungen billiger als Phase A. Die
DyCA+CVA-Fits fallen allerdings für jede Konfiguration erneut an (die
Projektion selbst lässt sich nicht aus Featurenamen sparen).

Die Selektion hat ausschließlich Trainingsdaten gesehen — das Testset bleibt
eine unverzerrte Generalisierungsschätzung.

In [ ]:
# ============================================================
# PHASE B: Testset - nur die in Phase A ausgewaehlten Features
# ============================================================
# Ergebnis liegt danach in pipe.test_top.
pipe.run_phase_b()

## Phase C — LazyClassifier je Konfiguration

Alle 7 Konfigurationen laufen mit demselben Klassifizierer-Satz. Zwei Dinge
sind bewusst so gesetzt (Begründungen wie im Schwester-Notebook):

**Identische Run-Menge.** Die DyCA-Stufe kann an einzelnen Runs numerisch
scheitern und verliert dann genau diese Runs. `restrict_to_common_runs=True`
schneidet deshalb alle Konfigurationen auf die Runs zu, die in **allen**
vorhanden sind — sonst würden die Setups auf unterschiedlichen (potenziell
unterschiedlich schweren) Testmengen bewertet.

**Macro-F1 als Kernzahl.** Balanced Accuracy ist der Macro-Recall und sieht
Precision nicht — sie belohnt Modelle, die eine Klasse als Sammelbecken
missbrauchen. Macro-F1 bestraft das. Beides wird berichtet; lazypredicts
eigene `F1 Score`-Spalte ist die *gewichtete* Variante und deshalb hier nicht
die Vergleichszahl.

**Modellauswahl über 5-fache CV — RandomForest bleibt der Hauptvergleich.**
Phase C läuft mit `lc_cv_folds = 5`: lazypredict kreuzvalidiert jedes Modell
zusätzlich auf den **Trainingsdaten** und liefert die Spalten `… CV Mean/Std`.
Die Auswahl „bestes Modell je Konfiguration" läuft darüber
(`BalancedAccCVMean`) und sieht das Testset nicht; berichtet werden weiterhin
die einmaligen Testwerte. Das alte Test-Maximum wird zum Vergleich mit
ausgegeben — es ist durch den Winner's Curse leicht optimistisch, und der
Abstand zwischen beiden zeigt, wie groß dieser Effekt hier ist.

Zwei Einschränkungen der CV:

- lazypredicts `F1 Score CV Mean` ist die **gewichtete** Variante; ein
  Macro-F1 aus der CV gibt es nicht. Selektionsmetrik ist deshalb
  `Balanced Accuracy CV Mean` — dieselbe Wahl wie `SELECT_METRIC` im
  Eigenwert-Notebook.
- Modelle **ohne `predict_proba`** (LinearSVC, Ridge, SGD, …) bekommen **keine**
  CV-Werte: lazypredict rechnet alle CV-Scorer gebündelt mit
  `error_score="raise"`, der ROC-AUC-Scorer braucht aber Wahrscheinlichkeiten —
  scheitert er, werden alle CV-Spalten dieses Modells geleert. Sie fallen damit
  aus der CV-Auswahl heraus (im Test-Maximum sind sie weiter dabei).

Der **RandomForest-Block** bleibt der belastbare Konfigurationsvergleich:
festes Modell, gar keine Auswahl.

> Kosten: die CV fittet jedes Modell fünfmal zusätzlich (Folds parallel,
> `n_jobs=-1`) — Phase C dauert grob das 2- bis 4-Fache. Die teuren Phasen A
> und B sind nicht betroffen und bleiben gecacht.

In [ ]:
# ============================================================
# PHASE C: LazyClassifier je Konfiguration
# ============================================================
# Schreibt die summary-CSV in den Cache-Ordner und legt sie zusaetzlich
# in pipe.summary ab. Die vollen Leaderboards stehen in
# pipe.leaderboards[name].
pipe.run_phase_c()

In [ ]:
# ============================================================
# Vergleich der Konfigurationen
# ============================================================
# Laeuft nach einem Kernel-Neustart auch OHNE Phase A/B/C: die summary
# liegt als CSV im Cache und wird von compare() nachgeladen. Vorher nur
# die Import-, Konfigurations- und Pipeline-Zelle ausfuehren.
cmp = pipe.compare()

In [ ]:
# ============================================================
# Plot: Hauptvergleich (RandomForest) + bestes Modell als Marker
# ============================================================
_ = pipe.plot_comparison(cmp)

## Confusion-Matrizen je Konfiguration

Dieselbe Auswertung wie am Ende von `LazyClassifier_PCA_DyCA.ipynb`, hier über
alle 7 Konfigurationen: **ein fester Modelltyp** — RandomForest, also der
Hauptvergleich von oben — wird pro Konfiguration auf dem **gesamten**
Trainingssatz gefittet und **einmal** auf dem echten Testset ausgewertet. Die
Unterschiede zwischen den Matrizen liegen damit allein an den Features.

**Warum neu gefittet wird.** Phase C berechnet die Vorhersagen zwar schon
(`predictions=True`), überschreibt `preds` aber je Konfiguration und hebt nichts
davon auf. Ein einzelner RF-Fit je Konfiguration auf `top_k = 100` Features
kostet Sekunden bis rund eine Minute — deutlich billiger, als Phase C mit ~25
Modellen zu wiederholen. Die Zelle gleicht am Ende gegen
`tsfresh_summary_dycvda.csv` ab, ob sie die dortige RandomForest-Zeile
reproduziert.

**Datenbasis identisch zu Phase C:** gemeinsame Runs
(`restrict_to_common_runs`), dieselbe NaN/inf-Behandlung, `StandardScaler` +
RandomForest wie lazypredict intern. Die gemeinsamen Runs werden hier neu
bestimmt — die Zelle läuft also auch ohne vorher gelaufenes Phase C, solange
`pipe.train_top`/`pipe.test_top` aus Phase A/B im Kernel liegen.

**Darstellung.** Die Matrizen sind **zeilenweise normiert** (Zeilensumme = 1):
Zelle (i, j) ist der Anteil der wahren Klasse *i*, der als *j* vorhergesagt
wurde, die Diagonale also der Recall. Alle Panels teilen sich die Skala 0…1 und
sind dadurch direkt vergleichbar. Die absoluten Zählwerte stehen als
21×21-DataFrame in `cm.counts[name]`.

> **Zum `raw`-Panel:** die Features sind bitidentisch zu denen in den
> Schwester-Notebooks, die *Runmenge* ist es nicht unbedingt:
> `restrict_to_common_runs` schneidet hier auf die Runs zu, die auch die
> DyCA-Stufe überlebt haben — je nachdem, welche Runs dort scheitern, sind die
> `raw`-Matrizen der drei Notebooks ähnlich, aber nicht notwendig identisch.

Die Vorhersagen werden als `tsfresh_cm_predictions_dycvda.csv` im Cache
abgelegt (eigener Name, damit die Dateien der Schwester-Notebooks im geteilten
`cache_dir` nicht überschrieben werden): nach einem Kernel-Neustart laufen die
Plotzellen darunter ohne Phase A/B/C. `refit=True` erzwingt die
Neuberechnung.

In [ ]:
# ============================================================
# Confusion-Matrizen: RandomForest je Konfiguration
# ============================================================
# refit=False nutzt den Vorhersage-Cache im Cache-Ordner; nach einem
# Kernel-Neustart laufen die Plotzellen darunter damit ganz ohne
# Phase A/B/C. refit=True fittet neu - noetig, wenn ueber estimator=...
# ein anderes Modell verglichen werden soll.
cm = pipe.confusion(refit=False)

In [ ]:
# ============================================================
# Plot: alle Confusion-Matrizen im Raster
# ============================================================
_ = pipe.plot_confusions(cm, ncols=4)

In [ ]:
# ============================================================
# Detail: eine Konfiguration gross + groesste Verwechslungen
# ============================================================
# focus=None waehlt die beste Konfiguration nach Macro-F1; sonst z.B.
# focus="raw". annot_min ist die Schwelle, ab der eine Zelle beschriftet
# wird, top_n die Laenge der Verwechslungsliste.
_, focus = plot_confusion_detail(cm, focus=None, annot_min=0.05, top_n=8)

cm.counts[focus]

In [ ]:
# ============================================================
# Recall je Fault-Klasse und Konfiguration
# ============================================================
recall_tab = plot_recall(cm)

recall_tab.round(3)

## Wo weitergemacht werden kann

- **Feature-Namen ansehen:** `pipe.top_names["dycvda_m6n12_s2_r12"]` zeigt,
  *welche* TSFresh-Features eine Konfiguration ausgewählt hat — und vor
  allem, aus welchen `cvd`-Kanälen sie stammen. Landet fast alles auf
  `cvd1`–`cvd3` (den stärksten kanonischen Korrelationen), trägt der
  Dissimilaritätsanteil der schwachen Kanäle wenig.
- **r-Verschachtelung prüfen:** `set(pipe.top_names["dycvda_m6n12_s2_r15"]) ==
  set(pipe.top_names["dycvda_m6n12_s2_r12"])` beantwortet direkt, ob die drei
  zusätzlichen Kanäle des Paper-Settings für die Top-100 überhaupt eine
  Rolle spielen (vgl. das `pca_6`/`pca_8`-Phänomen).
- **Quervergleich mit den Schwester-Notebooks:** `tsfresh_summary.csv`
  (PCA/DyCA), `tsfresh_summary_dpca_cva_ica.csv` (DPCA/CVA/ICA) und
  `tsfresh_summary_dycvda.csv` (dieses Notebook) liegen im selben
  Cache-Ordner und teilen die `raw`-Zeilen als gemeinsamen Anker — alle drei
  CSVs laden, konkatenieren und über sämtliche Konfigurationen plotten.
  Besonders interessant: `dycvda_m6n12_s2_r12` gegen `dyca_m6_n12` (gleiche
  DyCA-Stufe, mit vs. ohne CVA-Dissimilarität) und gegen `cva_12`
  (CVA-Idee auf Rohdaten vs. auf DyCA-Amplituden).
- **`top_k` variieren:** die Auswahl ist gecacht, ein anderer Wert erzwingt
  eine neue Selektion; die teure Train-Extraktion in `cache_dir` bleibt
  gültig. Die Phase-B-Test-Chunks heißen `*__test__top{top_k}__*.pkl` und
  kodieren den Wert — ein anderer `top_k` extrahiert das Testset sauber neu
  (Rechenzeit, keine stillen 0.0-Spalten).
- **CV wieder abschalten:** `lc_cv_folds = 0` in der Konfigurationszelle nimmt
  die 5-fache Train-CV aus Phase C heraus und spart grob das 2- bis 4-Fache der
  Phase-C-Zeit; die Modellauswahl fällt dann auf das Test-Maximum zurück. Die
  teuren Phasen A/B sind nicht betroffen und bleiben gecacht.
- **Parameter variieren:** neue (m, n, s, r)-Tupel in `DYCVDA_CONFIGS` sind
  gefahrlos — alle vier Parameter stecken im Konfigurationsnamen und damit im
  Cache-Präfix, ein Wechsel erzeugt neue Chunks statt alte stumm zu laden.
  Ausnahmen: `cva_ridge_rel` und `fix_signs` stehen nicht im Namen — bei
  Änderung vorher die `dycvda_*`-Chunks löschen.
- **Papiertreue Offline-Variante:** DyCA + CVA einmal auf den
  Fault-0-Trainingsruns fitten (Θ‡, J, L, Λ fest) und alle Runs mit dieser
  festen Projektion abbilden — das entspricht dem Offline/Online-Schema des
  Papers und macht die d-Kanäle über Runs hinweg physikalisch vergleichbar
  (lageabhängige TSFresh-Features werden dann uneingeschränkt sinnvoll).
  Optional zusätzlich den skalaren Monitoring-Index D(t) (Paper Gl. 16) als
  1-Kanal-Konfiguration mitlaufen lassen — der direkte Test, ob die
  Detektionsstatistik auch Klassifikationsinformation trägt.
- **Skalierung umstellen:** `scaling_mode` in der Konfigurationszelle
  (`"global_mean"` ↔ `"scaler"`); der Cache-Ordner bekommt den Modus
  automatisch angehängt, alte Ergebnisse werden nicht überschrieben.
- **Confusion-Matrizen:** das `cm`-Objekt aus der Zelle oben hält
  `cm.results` (Matrizen + Scores), `cm.counts` (absolute Zählwerte je
  Konfiguration) und `cm.recall_table()` (Recall je Klasse) bereit. Für
  ein anderes Modell `pipe.confusion(estimator=..., refit=True)` aufrufen
  — ohne `refit=True` wird der Vorhersage-Cache `tsfresh_cm_predictions_dycvda.csv`
  weiterverwendet.

- **Eigenes Verfahren ergänzen:** ein neuer `Projector` wird über
  `tep.tsfresh.register("mein_verfahren", Projector(...))` eingetragen und
  ist danach als Spec `("mein_verfahren", …)` in `CFG.configs` nutzbar —
  an der Pipeline selbst muss dafür nichts geändert werden. Vorbild sind
  die sieben Einträge in `tep/tsfresh/projections.py`; ein `Projector`
  braucht nur `name`, `channels` und `apply`.